# news_extract — 수집된 새 기사마다 원문 받기 → 숫자 영어 변환 → 리스트 추출

`news_collect.ipynb`가 구글 뉴스로 모은 기사 URL을 **DB(PostgreSQL)에 저장하면**, 새로 들어온 기사의 원문을 받고 LLM으로 정보를 뽑아(리스트 추출) `source/live/articles.jsonl`에 쌓는다. 쌓인 기사로 사건을 묶고 후속 보도·매체 수를 세는 일은 `news_track_tmp.ipynb`가 한다.

```
news_collect.ipynb  ─ 검색어마다 기사 URL을 DB(items·hits 표)에 쌓는다(읽기만 한다)
   ▼
news_extract_tmp.ipynb  ─ 새 기사 찾기 → 원문 받기 → 숫자 영어 변환 → 리스트 추출
   ▼  source/live/articles.jsonl, graph_source.jsonl, llm_source.jsonl
news_track_tmp.ipynb    ─ 사건 묶음 → 후속 보도 판별 → 매체 수
   ▼  source/live/stories.csv, followup.csv, importance.csv
```

| 절 | 하는 일 | 바탕이 된 코드 |
|---|---|---|
| 3 | 숫자 영어 변환(리스트 추출 입력에만) | proj `P1_기사조회_숫자변환_tmp.ipynb` 3절 |
| 4 | 원문 받기(거르기 2·3단계) | `news_process.ipynb` 5절을 규칙 그대로 |
| 5 | 새 기사 찾기(DB) | 새로 짬, 2026-11-20 수집이 DB로 바뀐 것에 맞춤 |
| 6 | 리스트 추출 | proj 시험에 쓴 OpenAI 추출, 프롬프트 `prompt/system_prompt_ver4.md` |

- **수집 노트북(`news_collect.ipynb`)과 그 DB는 고치지 않고 읽기만 한다.** 이 노트북이 쓰는 파일은 모두 `source/live/` 안에 있다(`source/`는 git에 올라가지 않는다).
- **본문은 오래 저장하지 않는다.** `articles.jsonl`에는 본문을 넣지 않고, 후속 판별에 쓸 사건 기사 본문만 `pending_bodies/<key>.txt`에 잠시 두었다가 `news_track_tmp.ipynb`가 쓴 뒤 지운다. 원문 HTML 캐시도 판정·추출이 끝나면 지운다.
- 처리한 기사는 `articles.jsonl`에 남으므로 멈췄다가 다시 켜도 처리하지 않은 기사부터 이어서 한다.

## 1. 설정

- `RUN_LLM = False`이면 새 기사 찾기·원문 받기·숫자 변환까지만 하고 **LLM을 부르지 않으며 결과를 저장하지 않는다**(받은 원문은 캐시에 남는다).
- `START_ROUND`: 이 회차부터 처리한다. 처음 켤 때 지난 회차 기사를 한꺼번에 추출하지 않게 한다(형식: `20261120T000000`, UTC). **DB에는 수집을 시작한 뒤의 기사가 모두 있어서, 오래된 값이면 그 뒤 기사를 한꺼번에 추출한다(비용).** 켤 때 그 시각 무렵으로 맞춘다.
- `DB_ENV`: 수집 DB 주소를 읽을 `.env` 이름(`GNEWS_DB_URL`, 읽기 전용 주소여도 된다).
- `CUTOFF_KST`: 수집 마감(한국 시각). 보고 날짜(`report`)의 경계다(전날 마감 뒤부터 그날 마감까지 처음 받은 기사가 그날). 표시 시각 08:00이면 07:30.
- `MAX_ARTICLES`: 한 번에 처리할 최대 기사 수(시험용). `None`이면 제한 없음.
- `EXTRACT_WORKERS`: 리스트 추출을 동시에 보내는 수(기본 8). 한 건에 10~20초라 한 건씩이면 밀린 기사(예: 하룻밤 약 1,000건)를 따라잡는 데 몇 시간 걸려서 2026-11-20에 더했다. 결과와 저장 칸은 같다.
- 키는 proj 폴더 `.env`의 `OPENAI_API_KEY`이며 값은 출력하지 않는다. `.env`는 노트북 폴더에서 위로 올라가며 처음 나오는 것을 쓴다.

In [ ]:
import datetime as dt
import hashlib
import json
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import urllib.error
import urllib.request
from pathlib import Path

import sqlalchemy as sa

import numpy as np
import pandas as pd
from IPython.display import display

# ===== 여기만 바꿔서 쓴다 =====
ROOT = Path.cwd()                      # news_search 폴더(이 노트북이 있는 폴더에서 연다)
PROJ = next((p for p in ROOT.parents if (p / '.env').exists()), ROOT.parents[0])   # .env가 있는 폴더(노트북 폴더에서 위로 올라가며 처음 나오는 곳)
RESULT_DIR = ROOT / 'source' / 'live'  # news_extract·news_track이 함께 쓰는 결과 폴더
DB_ENV = 'GNEWS_DB_URL'  # news_collect.ipynb의 수집 DB 주소(.env 이름, 읽기만)
EXTRACT_PROMPT_PATH = ROOT / 'prompt' / 'system_prompt_ver4.md'   # proj/prompt에서 복제
LLM_MODEL = 'gpt-6.1-terra'
RUN_LLM = False          # True여야 OpenAI로 리스트 추출을 한다
START_ROUND = '20261120T150000'   # 이 회차(UTC)부터 처리(11-21 00:00 한국, 11-21 아침 요약의 취침 시각 기본값). 오래된 값이면 지난 기사를 한꺼번에 추출
MAX_ARTICLES = None      # 한 번에 처리할 최대 기사 수(시험용)
EXTRACT_WORKERS = 8      # 리스트 추출을 동시에 보내는 수(2026-11-20). 1이면 한 건씩. 결과는 같고 밀린 기사를 빨리 따라잡는다
CUTOFF_KST = '07:30'     # 수집 마감(한국 시각). 보고 날짜의 경계(표시 시각 08:00이면 07:30)
WATCH_RUN = False        # True면 마지막 칸에서 WATCH_SECONDS초마다 DB의 새 기사를 확인하며 계속 처리
WATCH_SECONDS = 60
# ==============================

ARTICLES_PATH = RESULT_DIR / 'articles.jsonl'   # 기사마다 한 줄: 수집 정보, 원문 판정, 원문 본문, 추출 결과
PENDING_BODY_DIR = RESULT_DIR / 'pending_bodies'      # 후속 판별에 쓸 사건 기사 본문(임시, news_track이 쓴 뒤 지운다)
DELETE_HTML = True       # True면 판정·추출이 끝난 기사의 원문 HTML 캐시를 지운다(주소 복원·받기 기록은 남긴다)
LLM_SOURCE_PATH = RESULT_DIR / 'llm_source.jsonl'       # 추출한 기사마다 한 줄: 수집 정보 + 추출 결과 중 일부(LLM_SOURCE_FIELDS)
GRAPH_SOURCE_PATH = RESULT_DIR / 'graph_source.jsonl'   # 추출 성공한 기사마다 한 줄: key, graph(추출 결과의 graph)
USAGE_PATH = RESULT_DIR / 'llm_usage.csv'       # LLM 호출마다 토큰 수(news_track과 함께 씀)
EXTRACT_LOG_PATH = RESULT_DIR / 'extract_log.csv'   # 처리할 때마다 한 줄
RESULT_DIR.mkdir(parents=True, exist_ok=True)
assert ROOT.name == 'news_search' and (ROOT / 'news_collect.ipynb').exists(), 'news_search 폴더에서 열어야 한다.'

## 2. 작은 도구

- `env_value`: proj 폴더 `.env`에서 키를 읽는다(값은 출력하지 않는다).
- `ask_openai`: OpenAI Responses API. 오류가 나면 오류 종류·코드만 알린다(오류 본문에 가린 키 일부가 들어 있을 수 있어서). 잠깐 거절(429)·서버 오류(500·502·503)면 5초·15초 쉬고 다시 최대 3번.
- `log_usage`: 동시에 추출하므로 잠금을 걸고 llm_usage.csv에 한 줄씩 쓴다.

In [ ]:
def env_value(name):
    env_file = PROJ / '.env'
    if env_file.exists():
        for line in env_file.read_text(encoding='utf-8-sig').splitlines():
            key, _, value = line.partition('=')
            if key.strip() == name and value.strip():
                return value.strip().strip('"\'')
    return None


def read_jsonl(path):
    # 한 줄씩 읽는다. 다른 노트북이 쓰는 중인 마지막 줄(덜 쓰인 줄)은 건너뛴다.
    rows = []
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows


def append_jsonl(path, rows):
    with path.open('a', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


def append_csv(rows, path):
    if rows:
        pd.DataFrame(rows).to_csv(path, mode='a', header=not path.exists(), index=False, encoding='utf-8-sig')


class LLMError(Exception):
    pass


RETRY_HTTP = (429, 500, 502, 503)   # 잠깐 거절·서버 오류면 쉬고 다시(동시에 보내면서 2026-11-20 더함)


def ask_openai(instructions, user_message, text_format, tries=3):
    for k in range(tries):
        try:
            return _ask_openai(instructions, user_message, text_format)
        except LLMError as e:
            if k == tries - 1 or not any(f'HTTP {c} ' in str(e) for c in RETRY_HTTP):
                raise
            time.sleep(5 * 3 ** k)   # 5초, 15초


def _ask_openai(instructions, user_message, text_format):
    request = urllib.request.Request(
        'https://api.openai.com/v1/responses', method='POST',
        headers={'Authorization': f"Bearer {env_value('OPENAI_API_KEY')}", 'Content-Type': 'application/json'},
        data=json.dumps({'model': LLM_MODEL, 'instructions': instructions, 'input': user_message, 'store': False,
                         'reasoning': {'effort': 'medium'}, 'text': {'format': text_format}}).encode('utf-8'))
    try:
        with urllib.request.urlopen(request, timeout=600) as response:
            body = json.loads(response.read().decode('utf-8'))
    except urllib.error.HTTPError as e:
        try:
            err = json.loads(e.read().decode('utf-8', 'replace')).get('error', {})
        except ValueError:
            err = {}
        raise LLMError(f"OpenAI HTTP {e.code} type={err.get('type')} code={err.get('code')}") from None
    text = ''.join(c.get('text', '') for item in body.get('output', []) if item.get('type') == 'message'
                   for c in item.get('content', []))
    return text, body.get('usage') or {}


_USAGE_LOCK = threading.Lock()   # 여러 일꾼이 llm_usage.csv에 함께 쓴다


def log_usage(step, ref, usage):
    with _USAGE_LOCK:
        _log_usage(step, ref, usage)


def _log_usage(step, ref, usage):
    append_csv([{'time_utc': dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds'), 'step': step, 'ref': ref,
                 'input_tokens': usage.get('input_tokens'),
                 'cached_tokens': (usage.get('input_tokens_details') or {}).get('cached_tokens'),
                 'output_tokens': usage.get('output_tokens')}], USAGE_PATH)


print('OPENAI_API_KEY:', '설정됨' if env_value('OPENAI_API_KEY') else '미설정')

## 3. 숫자 영어 변환

`P1_기사조회_숫자변환_tmp.ipynb` 3절(`convert_news_numbers.ipynb`와 같은 규칙)을 그대로 옮겼다. **리스트 추출 입력에만 쓴다.**

- 10,000 이상인 숫자만 `num2words`로 영어 표기한다. 10,000 미만은 그대로 둔다.
- 10,000 이상 1,000,000 미만은 반올림 없이, 1,000,000 이상은 가장 가까운 백만 단위로 반올림한다.
- 숫자와 콜론이 있고 같은 문장에 월·요일·AM/PM 등 날짜 단서가 있으면 그 문장은 바꾸지 않는다.
- 종목코드(`KS:012450`, `005930.KS` 등)는 바꾸지 않는다.

In [ ]:
import re
from decimal import Decimal, ROUND_HALF_UP

from num2words import num2words

# 문자·기호와 붙어 있어도 숫자 표현을 찾되, 숫자 덩어리의 일부만 떼어내지 않는다.
NUMBER_PATTERN = re.compile(r"(?<!\d)(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?(?!\d)")
DATE_CUE_PATTERN = re.compile(
    r"\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?|Mon(?:day)?|Tue(?:sday)?|Wed(?:nesday)?|Thu(?:rsday)?|Fri(?:day)?|Sat(?:urday)?|Sun(?:day)?|AM|PM|EDT|EST|CDT|CST|MDT|MST|PDT|PST)\b",
    re.IGNORECASE,
)
COLON_NUMBER_PATTERN = re.compile(r"(?:\d\s*:|:\s*\d)")
# 종목코드는 금액이 아니므로 변환하지 않는다. 예: KS:012450, SZ:300136, 005930.KS
TICKER_PREFIX_PATTERN = re.compile(r"[A-Z]{1,10}:$")
TICKER_SUFFIX_PATTERN = re.compile(r"\.(?:KS|KQ|SS|SZ|HK|T|TW|TWO)\b")
SENTENCE_BOUNDARY = re.compile(r"((?<=[.!?])\s+(?=[A-Z])|\n+)")


def convert_number_token(raw):
    value = Decimal(raw.replace(",", ""))
    if abs(value) < Decimal(10_000):
        return raw
    if abs(value) < Decimal(1_000_000):
        return num2words(value, lang="en")
    rounded_millions = (value / Decimal(1_000_000)).quantize(Decimal("1"), rounding=ROUND_HALF_UP)
    return num2words(int(rounded_millions * Decimal(1_000_000)), lang="en")


def convert_numbers(text):
    """글 속의 숫자를 위 규칙대로 영어로 바꾼다."""
    parts = SENTENCE_BOUNDARY.split(text or "")
    converted_parts = []
    for part_index, sentence in enumerate(parts):
        if part_index % 2:   # 문장 사이 공백·줄바꿈
            converted_parts.append(sentence)
            continue
        if COLON_NUMBER_PATTERN.search(sentence) and DATE_CUE_PATTERN.search(sentence):
            converted_parts.append(sentence)
            continue

        def replace(match):
            raw = match.group(0)
            start, end = match.span()
            if TICKER_PREFIX_PATTERN.search(sentence[:start]) or TICKER_SUFFIX_PATTERN.match(sentence, end):
                return raw
            converted = convert_number_token(raw)
            if converted == raw:
                return raw
            # 영문자에 붙은 값은 변환 뒤 읽기 쉽도록 경계에 공백을 둔다.
            if start > 0 and sentence[start - 1].isalpha():
                converted = " " + converted
            if end < len(sentence) and sentence[end].isalpha():
                converted += " "
            return converted

        converted_parts.append(NUMBER_PATTERN.sub(replace, sentence))
    return "".join(converted_parts)


# 예: 첫 줄은 바뀌고(종목코드·10,000 미만은 그대로), 둘째 줄은 날짜·시각이 있는 문장이라 통째로 그대로 둔다.
print(convert_numbers("32520 / 12,345,678 / $8B / Hanwha Aerospace (KS:012450) / 9,999"))
print(convert_numbers("Nov. 17 at 4:30 PM, 25,000 shares were sold."))

## 4. 원문 받기 (거르기 2·3단계)

`news_process.ipynb` 5절의 본문 받기 코드를 **규칙을 바꾸지 않고** 옮겼다. 처리 노트북의 캐시(`source/bodies/`)를 건드리지 않도록 캐시는 `source/live/bodies/`에 따로 둔다.

```
구글 링크 → 원문 주소 복원(googlenewsdecoder) → 원문 페이지 받기 → 본문 추출(trafilatura) → 기사 판정(judge_page) → 본문
```

| 판정(`status`) | 뜻 |
|---|---|
| SUCCESS | 본문을 쓴다 |
| EXCLUDED_SECTION, QUOTE_PAGE | 거르기 2단계(원문 주소): 매체별 제외 구역, 야후 시세 페이지 |
| VIDEO_PAGE, LIVE_BLOG, PRESS_RELEASE, NOT_ARTICLE, PAYWALL, TOO_SHORT, PARTIAL_BODY | 거르기 3단계(페이지): 영상, 라이브, 보도자료, 기사 아님, 유료벽, 본문 짧음, 일부만 받음 |
| NOT_RESOLVED, HTTP_ERROR, TIMEOUT, BLOCKED, REDIRECTED_AWAY 등 | 받기 실패 |

- 원문 주소 복원도 구글 요청이다. 검색과 한도가 따로라고 시험했지만(언론사 시험 6,453건 모두 성공), 수집과 겹쳐 돌린 적은 없다.
- 원문 페이지는 매체마다 0.5초 간격, 매체끼리는 동시에 받는다.

In [ ]:
import asyncio
import concurrent.futures as cf
import datetime as dt
import gzip
import hashlib
import json
import re
import threading
import time
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests
from googlenewsdecoder import gnews_decoder_async
from lxml import html as lxml_html
from trafilatura import extract
from trafilatura.utils import decode_file

BODY_DIR = RESULT_DIR / 'bodies'   # 기사마다 한 번만 받는다
RESOLVE_CACHE = BODY_DIR / 'resolve_cache.jsonl'   # 구글 링크 → 원문 주소
FETCH_CACHE = BODY_DIR / 'fetch_cache.jsonl'       # 원문 페이지 받은 기록
HTML_DIR = BODY_DIR / 'html'                        # 원문 HTML(gzip)

USER_AGENT = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0 Safari/537.36'
RESOLVE_BATCH, RESOLVE_CONCURRENCY, RESOLVE_INTERVAL = 40, 4, 0.2   # 언론사 시험 때 6,453건을 막힘 없이 복원한 설정
FETCH_SLEEP = 0.5            # 같은 매체에 보내는 요청 사이 간격(초). 매체끼리는 동시에 받는다
FETCH_TIMEOUT = 15
MIN_BODY_WORDS = 80          # 본문이 이보다 짧으면 기사로 보지 않는다
MIN_LD_COVERAGE = 0.6        # JSON-LD 본문 대비 추출 본문 비율이 이보다 낮으면 일부만 받은 것

ARTICLE_LD_TYPES = {'Article', 'NewsArticle', 'ReportageNewsArticle', 'AnalysisNewsArticle', 'OpinionNewsArticle',
                    'BackgroundNewsArticle', 'ReviewNewsArticle', 'BlogPosting', 'Report'}
PAYWALL_PHRASES = ['subscribe to continue', 'subscribe to read', 'subscription required', 'to continue reading',
                   'already a subscriber', 'sign in to continue', 'sign in to read', 'log in to continue',
                   'create a free account to continue', 'this article is for subscribers', 'this content is for subscribers',
                   'unlock this article', 'become a member to read']
BLOCK_TITLE_PHRASES = ['access denied', 'just a moment', 'attention required', 'are you a robot', 'pardon our interruption',
                       'request unsuccessful', 'verify you are human', '403 forbidden']
BLOCK_HTML_MARKERS = ['px-captcha', 'captcha-delivery.com', 'cf-chl-', '_incapsula_resource']
PRESS_RELEASE_PROVIDERS = ['globenewswire', 'globe newswire', 'pr newswire', 'prnewswire', 'business wire', 'accesswire',
                           'access newswire', 'newsfile', 'eqs news']
PRESS_RELEASE_BODY = re.compile(r'PR ?Newswire|Business ?Wire|Globe ?Newswire|ACCESS ?Newswire|ACCESSWIRE|Newsfile|EQS-News|/CNW/', re.I)
# 거르기 2단계(원문 주소): 영상·라이브·시세 페이지, 매체별 제외 구역
VIDEO_PATH = re.compile(r'/(video|videos|watch|clip|clips|v|live-tv|livetv|podcast|podcasts|audio|gallery|galleries|slideshow|photos)(/|$)', re.I)
VIDEO_HOST = re.compile(r'^(video|videos|watch|tv)\.', re.I)
LIVE_PATH = re.compile(r'/(live|live-news|live-updates|liveblog)(/|$)|live-updates', re.I)
QUOTE_PATH = re.compile(r'finance\.yahoo\.com/quote/|/price-of-|/industry-statistics/', re.I)
SECTION_EXCLUDE_RULES = {
    'businessinsider.com': [r'^markets\.', r'^africa\.'], 'usatoday.com': [r'^ftw\.', r'^patriotswire\.', r'/story/shopping/', r'/press-release/'],
    'cnbc.com': [r'/investingclub/'], 'cnn.com': [r'^transcripts\.', r'/cnn-underscored/', r'^cnnicevents\.'], 'nypost.com': [r'/betting/'],
}

now_utc = lambda: dt.datetime.now(dt.timezone.utc)
host_of = lambda u: (urlparse(u or '').netloc or '').lower().removeprefix('www.')
url_hash = lambda u: hashlib.sha256((u or '').encode('utf-8')).hexdigest()[:24]


def _read_jsonl(path):
    return [json.loads(l) for l in path.read_text(encoding='utf-8').splitlines() if l.strip()] if path.exists() else []


def _append_jsonl(path, rows):
    if rows:
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open('a', encoding='utf-8') as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + '\n')


def resolve_urls(google_urls, log=print):
    """구글 뉴스 링크 → 원문 주소. 캐시에 없는 것만 googlenewsdecoder로 복원한다."""
    cache = {r['google_url']: r for r in _read_jsonl(RESOLVE_CACHE) if r.get('original_url')}
    todo = [u for u in dict.fromkeys(google_urls) if u not in cache]
    for i in range(0, len(todo), RESOLVE_BATCH):
        batch = todo[i:i + RESOLVE_BATCH]
        coro = lambda: gnews_decoder_async(batch, interval=RESOLVE_INTERVAL, timeout=15.0, concurrency=RESOLVE_CONCURRENCY)
        try:
            with cf.ThreadPoolExecutor(1) as ex:   # 노트북(이벤트 루프가 이미 도는 곳)에서도 쓰려고 별도 스레드에서 돌린다
                results = ex.submit(asyncio.run, coro()).result()
            results = [results] if isinstance(results, dict) else results
        except Exception as e:
            results = [{'success': False, 'message': f'{type(e).__name__}: {e}'}] * len(batch)
        rows = [dict(google_url=u, original_url=r.get('decoded_url') if r.get('success') else None,
                     error=None if r.get('success') else str(r.get('message'))[:200], resolved_utc=now_utc().isoformat())
                for u, r in zip(batch, results)]
        _append_jsonl(RESOLVE_CACHE, rows)
        cache.update({r['google_url']: r for r in rows})
    if todo:
        log(f'  원문 주소 복원 {len(todo)}건(성공 {sum(1 for u in todo if cache[u].get("original_url"))})')
    return {u: cache[u].get('original_url') for u in google_urls if u in cache}


def fetch_pages(urls, log=print):
    """원문 페이지 받기(캐시). 매체마다 FETCH_SLEEP 간격으로 차례로, 매체끼리는 동시에 받는다."""
    cache = {r['url']: r for r in _read_jsonl(FETCH_CACHE) if r.get('status') == 'OK'}
    # 받았던 페이지라도 원문 HTML 파일이 지워졌으면(DELETE_HTML) 다시 받는다. 안 그러면 같은 원문 주소의 다른 기사·다시 시도가 NOT_FETCHED가 된다(2026-11-22)
    cache = {u: r for u, r in cache.items() if (HTML_DIR / f'{url_hash(u)}.html.gz').exists()}
    todo = [u for u in dict.fromkeys(urls) if u and u not in cache]
    by_host = {}
    for u in todo:
        by_host.setdefault(host_of(u), []).append(u)
    HTML_DIR.mkdir(parents=True, exist_ok=True)
    lock = threading.Lock()
    out_rows = []

    def worker(host_urls):
        ses = requests.Session()
        ses.headers.update({'User-Agent': USER_AGENT,
                            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8', 'Accept-Language': 'en-US,en;q=0.9'})
        for u in host_urls:
            rec = dict(url=u, final_url=None, http=None, status=None, fetched_utc=now_utc().isoformat())
            try:
                r = ses.get(u, timeout=FETCH_TIMEOUT, allow_redirects=True)
                ctype = r.headers.get('Content-Type', '').lower()
                rec.update(final_url=r.url, http=r.status_code,
                           status='OK' if r.status_code == 200 and 'html' in ctype else ('NOT_HTML' if r.status_code == 200 else 'HTTP_ERROR'))
                if 'html' in ctype:
                    (HTML_DIR / f'{url_hash(u)}.html.gz').write_bytes(gzip.compress(r.content))
            except requests.RequestException as e:
                rec.update(status='TIMEOUT' if isinstance(e, requests.Timeout) else 'REQUEST_ERROR')
            with lock:
                out_rows.append(rec)
            time.sleep(FETCH_SLEEP)

    with cf.ThreadPoolExecutor(max_workers=max(1, min(12, len(by_host)))) as ex:
        list(ex.map(worker, by_host.values()))
    _append_jsonl(FETCH_CACHE, out_rows)
    cache.update({r['url']: r for r in out_rows})
    if todo:
        log(f'  원문 페이지 {len(todo)}건(매체 {len(by_host)}곳, 성공 {sum(r["status"] == "OK" for r in out_rows)})')
    return {u: cache.get(u) for u in urls if u}


def read_html(url):
    p = HTML_DIR / f'{url_hash(url)}.html.gz'
    return decode_file(gzip.decompress(p.read_bytes())) if p.exists() else None


def _ld_items(tree):
    items = []
    for node in tree.xpath('//script[@type="application/ld+json"]'):
        try:
            stack = [json.loads(node.text_content().strip())]
        except Exception:
            continue
        while stack:
            cur = stack.pop()
            if isinstance(cur, list):
                stack.extend(cur)
            elif isinstance(cur, dict):
                items.append(cur)
                if '@graph' in cur:
                    stack.append(cur['@graph'])
    return items


def _types(item):
    t = item.get('@type', [])
    return {x for x in ([t] if isinstance(t, str) else t) if isinstance(x, str)}


def page_features(raw_html, final_url):
    """원문 HTML → 판정에 쓰는 특징과 본문."""
    tree = lxml_html.fromstring(re.sub(r'^\s*<\?xml[^>]*\?>', '', raw_html))
    title = (tree.findtext('.//title') or '').strip()
    og = tree.xpath('//meta[@property="og:type"]/@content')
    og_type = og[0].strip().lower() if og else None
    items = _ld_items(tree)
    ld_types = set().union(*[_types(i) for i in items]) if items else set()
    arts = [i for i in items if _types(i) & ARTICLE_LD_TYPES]
    ld_words = max((len(str(i.get('articleBody') or '').split()) for i in arts), default=0)
    provider = ''
    for i in arts:
        for v in (i.get('provider'), i.get('sourceOrganization'), i.get('author')):
            v = v[0] if isinstance(v, list) and v else v
            if isinstance(v, dict) and v.get('@type') in (None, 'Organization', 'NewsMediaOrganization') and v.get('name'):
                provider = provider or v['name']
    published = next((i.get('datePublished') for i in arts if i.get('datePublished')), None) \
        or next(iter(tree.xpath('//meta[@property="article:published_time"]/@content')), None)
    body = (extract(raw_html, url=final_url, include_comments=False, include_tables=False, favor_precision=True) or '').strip()
    low = body.lower()
    return dict(page_title=title, og_type=og_type, ld_has_article=bool(arts), ld_has_video='VideoObject' in ld_types,
                ld_has_liveblog='LiveBlogPosting' in ld_types, ld_words=ld_words, provider=provider, published=published,
                blocked=any(p in title.lower() for p in BLOCK_TITLE_PHRASES) or any(m in raw_html[:50000].lower() for m in BLOCK_HTML_MARKERS),
                paywall=next((p for p in PAYWALL_PHRASES if p in low), None),
                press_release=any(n in provider.lower() for n in PRESS_RELEASE_PROVIDERS) or bool(PRESS_RELEASE_BODY.search(body[:600])),
                body=body, body_words=len(body.split()))


def judge_page(outlet, url, fetch, f):
    """원문 페이지 판정: 'SUCCESS'면 본문을 쓴다. 아니면 사유(거르기 2·3단계, 받기 실패)."""
    target = host_of(url) + urlparse(url or '').path
    if any(re.search(r, target) for r in SECTION_EXCLUDE_RULES.get(outlet, [])):
        return 'EXCLUDED_SECTION'
    if QUOTE_PATH.search(url or ''):
        return 'QUOTE_PAGE'
    if fetch is None or fetch.get('status') != 'OK':
        return (fetch or {}).get('status') or 'NOT_FETCHED'
    final = fetch.get('final_url') or url
    if not (host_of(final) == outlet or host_of(final).endswith('.' + outlet)):
        return 'REDIRECTED_AWAY'
    if f is None:
        return 'NOT_FETCHED'
    if f['blocked']:
        return 'BLOCKED'
    path = urlparse(final).path
    if VIDEO_PATH.search(path) or VIDEO_HOST.search(host_of(final)) or (f['og_type'] or '').startswith('video') \
            or (f['ld_has_video'] and not f['ld_has_article']):
        return 'VIDEO_PAGE'
    if LIVE_PATH.search(path) or f['ld_has_liveblog']:
        return 'LIVE_BLOG'
    if f['press_release']:
        return 'PRESS_RELEASE'
    if not f['ld_has_article'] or f['og_type'] == 'website' or not f['published']:
        return 'NOT_ARTICLE'
    if not f['body']:
        return 'EXTRACT_FAILED'
    if f['paywall']:
        return 'PAYWALL'
    if f['body_words'] < MIN_BODY_WORDS:
        return 'TOO_SHORT'
    if f['ld_words'] and f['body_words'] / f['ld_words'] < MIN_LD_COVERAGE:
        return 'PARTIAL_BODY'
    return 'SUCCESS'


def get_bodies(items, log=print):
    """기사 표(key, link, outlet) → 본문 표: key, original_url, final_url, status, body, body_words, page_title, provider."""
    orig = resolve_urls(items['link'].tolist(), log=log)
    fetch = fetch_pages([orig.get(u) for u in items['link']], log=log)
    rows = []
    for r in items.itertuples():
        url = orig.get(r.link)
        fr = fetch.get(url) if url else None
        raw = read_html(url) if fr and fr.get('status') == 'OK' else None
        try:
            f = page_features(raw, fr['final_url']) if raw else None
        except Exception:
            f = None
        status = 'NOT_RESOLVED' if not url else judge_page(r.outlet, url, fr, f)
        rows.append(dict(key=r.key, original_url=url, final_url=(fr or {}).get('final_url'), status=status,
                         body=f['body'] if f and status == 'SUCCESS' else '', body_words=f['body_words'] if f else 0,
                         page_title=f['page_title'] if f else '', provider=f['provider'] if f else ''))
    return pd.DataFrame(rows, columns=['key', 'original_url', 'final_url', 'status', 'body', 'body_words', 'page_title', 'provider'])

## 5. 새 기사 찾기 (DB)

수집 노트북이 쌓는 DB(`.env`의 `GNEWS_DB_URL`)에서 새 기사를 찾는다(2026-11-20, 보고 날짜 폴더에서 바꿈).

- 수집은 회차 없이 검색어마다 계속 검색하고, **검색어 하나의 검색(요청 3건: 야후·묶음 A·묶음 B)을 끝낼 때마다 그 결과를 한 번에 DB에 저장한다.** 그래서 DB에 보이는 기사는 이미 다 받은 것이고, 회차가 끝나기를 기다리지 않고 바로 처리한다.
- 기사의 **회차** = 그 기사를 처음 찾은 검색(`hits.round` 중 가장 이른 것, 형식 `20261120T013012`, UTC). 한 검색어의 3건에서 나온 기사가 한 회차라 `news_track`에서 함께 판단된다.
- **보고 날짜**(`report`) = 처음 받은 시각(`first_seen_utc`)이 `CUTOFF_KST` 뒤면 다음 날, 그 전이면 그날(예: 마감 07:30이면 11-19 23:00 → `20261120`, 11-20 07:40 → `20261121`).
- 거르기 1단계(`drop`)를 통과하고, `source/live/articles.jsonl`에 없는 기사만 고른다. `START_ROUND`보다 앞 회차는 빼고, 회차·발행 시각 순서로 처리한다.
- 대기하며 돌릴 때(8절)는 처리한 기사 목록을 메모리에 두고 새로 처리한 것만 더한다(`articles.jsonl` 전체를 매번 다시 읽지 않게). 처리 중 오류가 나면 다음에 파일에서 다시 읽는다.

In [ ]:
_ENGINE = None
_DONE = None     # 처리한 기사 key(articles.jsonl). 처음 한 번 읽고, 처리할 때마다 더한다.
# 받기 실패 다시 시도(2026-11-22): 마지막 판정이 아래 받기 실패면 처리 끝으로 보지 않고, 마지막 시도 RETRY_AFTER_MIN분 뒤 다시 처리한다(처음 포함 최대 1 + RETRY_MAX번).
# 새 기사가 먼저이고, 다시 시도는 한 바퀴에 RETRY_PER_ROUND건까지만 뒤에 붙인다(새 기사가 늦어지지 않게).
RETRY_STATUS = {'NOT_FETCHED', 'NOT_RESOLVED', 'HTTP_ERROR'}
RETRY_AFTER_MIN = 15
RETRY_MAX = 2
RETRY_PER_ROUND = 10
_TRIES = {}      # key → 지금까지 처리한 횟수(articles.jsonl 줄 수)
_LAST_TRY = {}   # key → 이 실행에서 마지막으로 처리한 시각(time.time())


def db_engine():
    global _ENGINE
    if _ENGINE is None:
        url = env_value(DB_ENV)
        if not url:
            raise RuntimeError(f'.env에 {DB_ENV}(수집 DB 주소)가 없다.')
        _ENGINE = sa.create_engine(url, pool_pre_ping=True)
    return _ENGINE


def done_keys():
    global _DONE
    if _DONE is None:
        last = {}
        _TRIES.clear()
        for r in read_jsonl(ARTICLES_PATH):
            last[r['key']] = r.get('status')
            _TRIES[r['key']] = _TRIES.get(r['key'], 0) + 1
        _DONE = {k for k, s in last.items() if s not in RETRY_STATUS or _TRIES[k] > RETRY_MAX}   # 받기 실패이고 횟수가 남은 기사는 다시 시도 대상
    return _DONE


def iso_utc(values):
    t = pd.to_datetime(values, utc=True, errors='coerce')
    return [x.isoformat() if pd.notna(x) else '' for x in t]


def report_of(first_seen):
    """처음 받은 시각 → 보고 날짜(YYYYMMDD). 수집 마감(CUTOFF_KST) 뒤면 다음 날."""
    h, m = map(int, CUTOFF_KST.split(':'))
    shift = pd.Timedelta(hours=h, minutes=m) + pd.Timedelta(microseconds=1)
    kst = pd.to_datetime(first_seen, utc=True).dt.tz_convert('Asia/Seoul')
    return ((kst - shift).dt.date + dt.timedelta(days=1)).map(lambda d: d.strftime('%Y%m%d'))


ITEMS_SQL = sa.text("""
    select i.key, i.title, i.source, i.domain, i.outlet, i.published_utc, i.link, i.first_seen_utc, i."drop",
           min(h.round) as round
    from items i join hits h on h.key = i.key
    where coalesce(i."drop", '') = '' and i.first_seen_utc >= :since
    group by i.key
    having min(h.round) >= :start_round
""")


def find_new_articles():
    start_round = START_ROUND or '00000000T000000'
    since = pd.to_datetime(start_round, format='%Y%m%dT%H%M%S', utc=True)   # 회차 시각 이후에 처음 받은 기사만(색인으로 빨리 거른다)
    with db_engine().connect() as con:
        items = pd.read_sql(ITEMS_SQL, con, params={'since': since.to_pydatetime(), 'start_round': start_round})
    items = items[~items['key'].isin(done_keys())]
    if items.empty:
        return pd.DataFrame()
    items['report'] = report_of(items['first_seen_utc'])
    for c in ['published_utc', 'first_seen_utc']:
        items[c] = iso_utc(items[c])                 # 폴더(CSV) 때와 같은 ISO 글자(UTC)
    items = items.fillna({'drop': ''}).sort_values(['round', 'published_utc']).reset_index(drop=True)
    retry = items['key'].isin(_TRIES.keys())                 # 전에 처리했지만 받기 실패라 다시 시도할 기사
    wait_ok = items['key'].map(lambda k: time.time() - _LAST_TRY.get(k, 0) >= RETRY_AFTER_MIN * 60)
    items = pd.concat([items[~retry], items[retry & wait_ok].head(RETRY_PER_ROUND)], ignore_index=True)   # 새 기사 먼저
    return items.head(MAX_ARTICLES) if MAX_ARTICLES else items


def round_span(rounds):
    r = sorted(set(rounds))
    return r[0] if len(r) == 1 else f'{r[0]} ~ {r[-1]} ({len(r)}개)'


new_preview = find_new_articles()
print(f'처리할 새 기사 {len(new_preview)}건' + (f' (회차 {round_span(new_preview["round"])})' if len(new_preview) else ''))
if len(new_preview):
    display(new_preview.groupby('report').agg(기사_수=('key', 'size'), 회차_수=('round', 'nunique'),
                                              처음_받은_시각=('first_seen_utc', 'min'), 마지막=('first_seen_utc', 'max')))

## 6. 리스트 추출 (원문 받기 → 숫자 영어 변환 → 추출)

- 원문 판정(`status`)이 `SUCCESS`인 기사만 추출한다. 판정이 다른 기사도 처리한 기사로 남겨 다시 받지 않는다.
- **숫자 영어 변환은 리스트 추출 입력에만 쓴다.** 후속 보도 판별에는 원문 그대로를 쓴다.
- **본문은 `articles.jsonl`에 저장하지 않는다.** 추출 결과가 사건 기사(`news_kind = event`, `comparison_summary` 있음)이면 원문 본문을 `pending_bodies/<key>.txt`에 잠시 둔다. 후속 판별에 쓰이면 `news_track_tmp.ipynb`가 지운다. 분석·추천 글, 추출 실패, 원문 판정이 SUCCESS가 아닌 기사는 본문을 남기지 않는다.
- `DELETE_HTML = True`면 기사 하나의 판정·추출이 끝날 때 원문 HTML 캐시(`bodies/html/`)를 지운다.
- 추출 입력은 시험 때와 같은 모양(`article_id, title, source, url, published_at, content`)이고, 출력은 JSON 객체로 받는다.
- 기사마다 추출이 끝나는 즉시 `articles.jsonl`에 한 줄씩 붙인다(중간에 멈춰도 낸 비용이 남는다). 추출은 `EXTRACT_WORKERS`건씩 동시에 보내고 끝나는 순서대로 붙인다(2026-11-20). 파일 줄 순서는 처리 순서와 다를 수 있지만 `news_track`은 회차·발행 시각으로 다시 정렬해서 결과는 같다.
- 추출 결과에 `graph`가 있으면 `{key, graph}`를 `graph_source.jsonl`에도 한 줄 붙인다.
- 추출을 시도한 기사(원문 `SUCCESS`, 추출 성공·실패 모두)는 `llm_source.jsonl`에도 한 줄 붙인다. 칸은 `LLM_SOURCE_FIELDS`: 수집 정보(key, first_seen_utc, first_seen_kst, published_utc, published_kst, round, report, title, outlet, provider, link, original_url), 추출 결과에서 article_type(article_profile 안, 거시 유형만 한국어로: `MACRO_TYPE_KO`, 없으면 빈 목록), entities·sectors(graph 안), company_impact, short_summary(article_profile 안), comparison_summary, 그리고 extract_error. 추출이 실패하면 추출 칸은 빈 값이다.
- `articles.jsonl`·`llm_source.jsonl`의 시각은 UTC(`*_utc`) 바로 뒤에 한국 시각(`*_kst`, `+09:00`)을 함께 둔다. 한국 시각이 없는 옛 기록으로 `llm_source.jsonl`을 만들 때는 UTC에서 계산한다.
- `build_llm_source()`: 이미 쌓인 `articles.jsonl`에서 `llm_source.jsonl`을 처음부터 다시 만든다.

In [ ]:
EXTRACT_PROMPT = EXTRACT_PROMPT_PATH.read_text(encoding='utf-8')


def extract_article(row):
    article = {'article_id': row['key'], 'title': convert_numbers(row['title']), 'source': row['outlet'],
               'url': row['original_url'], 'published_at': row['published_utc'], 'content': convert_numbers(row['body'])}
    # json_object 형식은 입력에 'json'이라는 단어가 있어야 한다.
    text, usage = ask_openai(EXTRACT_PROMPT, 'article json:\n' + json.dumps(article, ensure_ascii=False), {'type': 'json_object'})
    log_usage('extract', row['key'], usage)
    return json.loads(text)


LLM_SOURCE_FIELDS = ['key', 'first_seen_utc', 'first_seen_kst', 'published_utc', 'published_kst', 'round', 'report', 'title', 'outlet',
                     'provider', 'link', 'original_url',
                     'article_type', 'entities', 'sectors', 'company_impact', 'short_summary', 'comparison_summary', 'extract_error']

KST = dt.timezone(dt.timedelta(hours=9))


def to_kst(utc_text):
    """UTC ISO 시각 문자열 → 한국 시각 ISO 문자열(예: 2026-11-17T10:51:40+09:00). 비었거나 못 읽으면 빈 값."""
    try:
        when = dt.datetime.fromisoformat(str(utc_text).replace('Z', '+00:00'))
        if when.tzinfo is None:          # 시간대 표시가 없으면 UTC로 본다
            when = when.replace(tzinfo=dt.timezone.utc)
        return when.astimezone(KST).isoformat(timespec='seconds')
    except ValueError:
        return ''


# article_profile.article_type 중 거시 유형만 한국어로 남긴다(프롬프트 'article_type 정의'의 거시 12종).
MACRO_TYPE_KO = {'interest_rates': '금리', 'inflation': '물가', 'employment': '고용', 'economic_growth': '경기',
                 'fx': '환율', 'energy': '에너지', 'metals': '금속', 'agriculture': '농산물', 'trade_tariffs': '무역·관세',
                 'fiscal_industrial_policy': '재정·산업정책', 'geopolitics': '지정학', 'financial_stability': '금융 안정'}


def llm_source_row(record):
    """articles.jsonl 한 줄 → llm_source.jsonl 한 줄(LLM_SOURCE_FIELDS 순서)."""
    e = record.get('extraction') or {}
    graph, profile = e.get('graph') or {}, e.get('article_profile') or {}
    row = {k: record.get(k) for k in LLM_SOURCE_FIELDS[:12]}
    row['first_seen_kst'] = record.get('first_seen_kst') or to_kst(record.get('first_seen_utc'))
    row['published_kst'] = record.get('published_kst') or to_kst(record.get('published_utc'))
    row.update(article_type=[MACRO_TYPE_KO[t] for t in profile.get('article_type') or [] if t in MACRO_TYPE_KO],
               entities=graph.get('entities', []), sectors=graph.get('sectors', []), company_impact=e.get('company_impact', []),
               short_summary=profile.get('short_summary', ''), comparison_summary=e.get('comparison_summary', ''),
               extract_error=record.get('extract_error'))
    return row


def build_llm_source(articles_path=None, out_path=None):
    """쌓인 articles.jsonl에서 llm_source.jsonl을 처음부터 다시 만든다(추출을 시도한 기사만)."""
    articles_path, out_path = articles_path or ARTICLES_PATH, out_path or LLM_SOURCE_PATH
    rows = [llm_source_row(r) for r in read_jsonl(articles_path) if r.get('extraction') is not None or r.get('extract_error')]
    out_path.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in rows), encoding='utf-8')
    print(f'llm_source: {len(rows)}건 →', out_path)
    return len(rows)


def fetch_and_extract(new):
    bodies = get_bodies(new[['key', 'link', 'outlet']])
    new = new.merge(bodies, on='key', how='left')
    print('  원문 판정:', new['status'].value_counts().to_dict())
    ok = new['status'] == 'SUCCESS'
    if not RUN_LLM:
        print(f'  추출할 기사 {ok.sum()}건 — RUN_LLM = False라 추출하지 않고 저장하지 않는다')
        return new
    def save(r, extraction, error):
        """기사 하나를 저장한다(원래 반복문과 같은 칸·같은 파일)."""
        record = {c: r[c] for c in ['key', 'report', 'round', 'title', 'outlet', 'published_utc', 'first_seen_utc', 'link',
                                    'original_url', 'status', 'provider']}
        record = {**{k: v for k, v in record.items() if k not in ('first_seen_utc', 'link', 'original_url', 'status', 'provider')},
                  'published_kst': to_kst(record['published_utc']), 'first_seen_utc': record['first_seen_utc'],
                  'first_seen_kst': to_kst(record['first_seen_utc']),
                  **{k: record[k] for k in ('link', 'original_url', 'status', 'provider')}}   # utc 바로 뒤에 한국 시각
        record.update(extraction=extraction, extract_error=error)    # 본문은 저장하지 않는다
        e = record['extraction'] or {}
        if (e.get('article_profile') or {}).get('news_kind') == 'event' and (e.get('comparison_summary') or '').strip():
            PENDING_BODY_DIR.mkdir(parents=True, exist_ok=True)   # 후속 판별용 본문(임시)
            (PENDING_BODY_DIR / f"{r['key']}.txt").write_text(r['body'], encoding='utf-8')
        append_jsonl(ARTICLES_PATH, [record])
        if DELETE_HTML and isinstance(r['original_url'], str):
            (HTML_DIR / f"{url_hash(r['original_url'])}.html.gz").unlink(missing_ok=True)
        if r['status'] == 'SUCCESS':
            append_jsonl(LLM_SOURCE_PATH, [llm_source_row(record)])
        if (record['extraction'] or {}).get('graph') is not None:
            append_jsonl(GRAPH_SOURCE_PATH, [{'key': record['key'], 'graph': record['extraction']['graph']}])

    def work(r):
        try:
            return extract_article(r), None
        except (LLMError, json.JSONDecodeError, OSError) as e:
            return None, f'{type(e).__name__}: {str(e)[:150]}'

    for _, r in new[~ok].iterrows():          # 원문을 못 받은 기사: 추출 없이 처리한 기사로 남긴다
        save(r, None, None)
    # 원문을 받은 기사: EXTRACT_WORKERS건씩 동시에 추출하고, 끝나는 대로 저장(쓰기는 이 반복문 하나에서만 → 파일이 섞이지 않음)
    pool = ThreadPoolExecutor(max_workers=max(1, EXTRACT_WORKERS))
    try:
        futures = {pool.submit(work, r): r for _, r in new[ok].iterrows()}
        for f in as_completed(futures):
            r = futures[f]
            extraction, error = f.result()
            if error:
                print('  추출 실패', r['key'], error)
            save(r, extraction, error)
    except BaseException:
        pool.shutdown(wait=False, cancel_futures=True)   # 멈추면 아직 보내지 않은 추출은 보내지 않는다(비용)
        raise
    pool.shutdown()
    print(f'  추출 {ok.sum()}건 저장 →', ARTICLES_PATH.name, ',', LLM_SOURCE_PATH.name, ',', GRAPH_SOURCE_PATH.name)
    return new

## 7. 한 번 처리하기

DB에 새 기사가 있으면 원문 받기·숫자 변환·리스트 추출을 하고, `extract_log.csv`에 한 줄 남긴다(`rounds`: 이번에 처리한 회차, 여러 개면 `처음 ~ 마지막 (n개)`).

In [ ]:
def run_once():
    global _DONE
    start = time.time()
    new = find_new_articles()
    if new.empty:
        return
    now = dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds')
    rounds = round_span(new['round'])
    print(f'[{now}] 새 기사 {len(new)}건 (회차 {rounds})')
    try:
        new = fetch_and_extract(new)
    except Exception:
        _DONE = None          # 중간에 멈췄으면 다음에 articles.jsonl에서 다시 읽는다
        raise
    if RUN_LLM:
        for k, s in zip(new['key'], new['status']):
            _TRIES[k] = _TRIES.get(k, 0) + 1
            _LAST_TRY[k] = time.time()
            if s not in RETRY_STATUS or _TRIES[k] > RETRY_MAX:   # 성공했거나 횟수를 다 쓴 기사만 처리 끝
                done_keys().add(k)
        append_csv([{'time_utc': now, 'rounds': rounds, 'new_articles': len(new),
                     'body_ok': int((new['status'] == 'SUCCESS').sum()), 'seconds': round(time.time() - start, 1)}], EXTRACT_LOG_PATH)


run_once()

## 8. 대기하며 돌리기

`WATCH_RUN = True`로 켜 두면 `WATCH_SECONDS`초마다 DB의 새 기사를 확인해 처리한다. 수집(`news_collect.ipynb`), `news_track_tmp.ipynb`와 함께 켜 둔다. 오류가 나도 멈추지 않고 다음에 다시 한다. 멈추려면 커널을 중단한다(■).

In [ ]:
if WATCH_RUN:
    print(f'{WATCH_SECONDS}초마다 DB의 새 기사를 확인합니다. 멈추려면 커널을 중단하세요.')
    while True:
        try:
            run_once()
        except Exception as e:
            print('오류:', type(e).__name__, str(e)[:200], '(다음에 다시 시도)')
        time.sleep(WATCH_SECONDS)
else:
    print('WATCH_RUN = False: 대기하며 돌리지 않는다.')